In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
train_dir = r"C:\Users\liann\Desktop\train"
val_dir = r"C:\Users\liann\Desktop\test"

In [5]:
batch_size = 32
epochs = 5
learning_rate = 1e-3

#transforming/normalizing for model use
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), #more variety by flipping some photos
    transforms.ToTensor(), #pixels
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

#assigns 0-3 for each food category
train_ds = datasets.ImageFolder(train_dir, transform=train_tfms) 
val_ds = datasets.ImageFolder(val_dir, transform=val_tfms)

train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

#counts number of food classes (4 for now)
num_classes = len(train_ds.classes)

print("Classes:", train_ds.classes)
print("Class mapping:", train_ds.class_to_idx)
print("Train images:", len(train_ds))
print("Validation images:", len(val_ds))

###MODEL###
#pretrained resnet model on other images
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

#keeps pretrained knowledge such as edges, shapes, textures
for param in model.parameters():
    param.requires_grad = False

#so it only uses 4, not the other classes in the pretrained model
model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, num_classes)
)

model = model.to(device)

loss_fn = nn.CrossEntropyLoss() #measures prediction error
optimizer = torch.optim.Adam(model.fc.parameters(), lr=learning_rate) #updates model weights

###TRAININGFUNCTION### runs one epoch
def train_one_epoch():
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_dl:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images) #forward pass
        loss = loss_fn(outputs, labels) #measures prediction error

        loss.backward() #backpropagation
        optimizer.step() #updates weights to improve model

        total_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

###VALIDATIONFUNCTION### tests model on unseen images to measure accuracy
@torch.no_grad()
def evaluate():
    model.eval()

    correct = 0
    total = 0

    for images, labels in val_dl:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    accuracy = correct / total
    return accuracy

###TRAININGLOOP### runs multiple epochs to increase accuracy
for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch()
    val_acc = evaluate()

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

#save the model
torch.save({
    "model_state_dict": model.state_dict(),
    "classes": train_ds.classes
}, "food_model.pth")

print("Model saved as food_model.pth")

Classes: ['cheesecake', 'hamburger', 'omelette', 'pizza']
Class mapping: {'cheesecake': 0, 'hamburger': 1, 'omelette': 2, 'pizza': 3}
Train images: 3600
Validation images: 400
Epoch 1/5 | Train Loss: 0.7059 | Train Acc: 0.7386 | Val Acc: 0.8475
Epoch 2/5 | Train Loss: 0.4026 | Train Acc: 0.8544 | Val Acc: 0.8975
Epoch 3/5 | Train Loss: 0.3541 | Train Acc: 0.8764 | Val Acc: 0.8925
Epoch 4/5 | Train Loss: 0.3466 | Train Acc: 0.8731 | Val Acc: 0.8925
Epoch 5/5 | Train Loss: 0.3255 | Train Acc: 0.8858 | Val Acc: 0.8950
Model saved as food_model.pth


In [7]:
###PREDICTS ONE NEW IMAGE###
from PIL import Image

def predict_image(image_path):
    model.eval()

    image = Image.open(image_path).convert("RGB")
    image = val_tfms(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image)
        probs = torch.softmax(outputs, dim=1)
        confidence, predicted_idx = torch.max(probs, dim=1)

    predicted_class = train_ds.classes[predicted_idx.item()]
    confidence = confidence.item()

    return predicted_class, confidence

image_path = r"C:\Users\liann\Downloads\download (11).jpg"

pred_class, confidence = predict_image(image_path)
print(f"Prediction: {pred_class}")
print(f"Confidence: {confidence:.2%}")

Prediction: cheesecake
Confidence: 96.41%


In [8]:
#edit train_dir and val_dir to use a different food dataset
#the classes are created from the folder names so make sure to name them what the food is
#for the website/app use predict_image function and change image_path to uploaded file, shouldn't be a file path